**Logistic Regression CPU**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import time as StopWatch
from numba import cuda, float32
import math

In [ ]:
Dataset = pd.read_csv('titanic.csv')

Dataset["Age"] = pd.to_numeric(Dataset["Age"], errors='coerce')
Dataset["Fare"] = pd.to_numeric(Dataset["Fare"], errors='coerce')
Dataset = Dataset.dropna(subset=['Age', 'Fare'])

In [ ]:
Dataset = pd.concat([Dataset]*3000, ignore_index=True)

In [ ]:
X = Dataset[['Pclass', 'Age', 'Fare']].values.astype(np.float32)
Y = Dataset['survived'].values.astype(np.float32)

Num = len(X)

print("Num Of Samples: ", Num)

In [ ]:
Scaler = StandardScaler()
XScaled = Scaler.fit_transform(X).astype(np.float32)

In [ ]:
def Sigmoid(z):
    return 1 / (1 + np.exp(-z))

def LogisticRegressionCPU(X, Y, epochs=1000, LearningRate=0.1):
    Records, Features = X.shape
    Weights = np.zeros(Features, dtype=np.float32)
    Bias = 0.0

    for i in range(epochs):

        Z = np.dot(X, Weights) + Bias
        YPredict = Sigmoid(Z)

        Error = YPredict - Y

        WeightDerivative = (1 / Records) * np.dot(X.T, Error)
        BiasDerivative = (1 / Records) * np.sum(Error)

        Weights -= LearningRate * WeightDerivative
        Bias -= LearningRate * BiasDerivative

    return Weights, Bias

In [ ]:
StartCPUTime = StopWatch.time()
Weights , Bias = LogisticRegressionCPU(XScaled,Y)
EndCPUTime = StopWatch.time()

CPUTime = EndCPUTime - StartCPUTime

Z = np.dot(XScaled,Weights) + Bias
YPredProb = Sigmoid(Z)
YPred = (YPredProb > 0.5).astype(np.float32)
Accuracy = accuracy_score(Y,YPred) * 100

print("Accuracy: {:.2f}%".format(Accuracy))
print("CPU Time: {:.2f} seconds".format(CPUTime))

**Logistic Regression GPU**

In [ ]:
@cuda.jit
def LogisticRegressionGPU(X, Y, Weights, Bias, dW, dB, Records):

    idx = cuda.grid(1)

    if idx < Records:

        Z = 0.0

        for j in range(X.shape[1]):
            Z += X[idx, j] * Weights[j]

        Z += Bias[0]

        YPredict = 1.0 / (1.0 + math.exp(-Z))
        Error = YPredict - Y[idx]

        for j in range(X.shape[1]):
            cuda.atomic.add(dW, j, X[idx, j] * Error)

        cuda.atomic.add(dB, 0, Error)

In [ ]:
XGPU = cuda.to_device(XScaled)
YGPU = cuda.to_device(Y)

WeightsGPU = cuda.to_device(np.zeros(XScaled.shape[1], dtype=np.float32))
BiasGPU = cuda.to_device(np.array([0.0], dtype=np.float32))

dWGPU = cuda.to_device(np.zeros(XScaled.shape[1], dtype=np.float32))
dBGPU = cuda.to_device(np.array([0.0], dtype=np.float32))

ThreadsPerBlock = 256
BlocksPerGrid = (Num + ThreadsPerBlock - 1) // ThreadsPerBlock

LearningRate = 0.1
epochs = 1000

StartGPUTime = StopWatch.time()

for epoch in range(epochs):

    dWGPU.copy_to_device(np.zeros(XScaled.shape[1], dtype=np.float32))
    dBGPU.copy_to_device(np.array([0.0], dtype=np.float32))

    LogisticRegressionGPU[BlocksPerGrid, ThreadsPerBlock](
        XGPU, YGPU,
        WeightsGPU, BiasGPU,
        dWGPU, dBGPU,
        Num
    )

    cuda.synchronize()

    dW = dWGPU.copy_to_host()
    dB = dBGPU.copy_to_host()[0]

    Weights = WeightsGPU.copy_to_host()
    Bias = BiasGPU.copy_to_host()[0]

    Weights -= LearningRate * (dW / Num)
    Bias -= LearningRate * (dB / Num)

    WeightsGPU = cuda.to_device(Weights)
    BiasGPU = cuda.to_device(np.array([Bias], dtype=np.float32))

cuda.synchronize()

EndGPUTime = StopWatch.time()

In [ ]:
Weights = WeightsGPU.copy_to_host()
Bias = BiasGPU.copy_to_host()[0]

Z = np.dot(XScaled, Weights) + Bias
YPredProb = Sigmoid(Z)
YPred = (YPredProb > 0.5).astype(np.float32)

Accuracy = accuracy_score(Y, YPred) * 100

print("GPU Accuracy: {:.2f}%".format(Accuracy))
print("GPU Time: {:.2f} sec".format(EndGPUTime - StartGPUTime))